# Generación real por planta

In [0]:
%pip install pydataxm

In [0]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

from pydataxm.pydatasimem import ReadSIMEM


bronze_table = "observatorio_dev.bronze.generacion_real"

historical_start_date = date(2026, 1, 1)
lookback_days = 45

# Fecha actual en Colombia
fecha_fin = datetime.now(
    ZoneInfo("America/Bogota")
).date()

# Revisar si Bronze está vacía sin contar toda la tabla
bronze_is_empty = len(
    spark.table(bronze_table).head(1)
) == 0

if bronze_is_empty:
    # Primera ejecución
    fecha_inicio = historical_start_date
    execution_mode = "BACKFILL"
else:
    # Ejecuciones diarias posteriores
    fecha_inicio = fecha_fin - timedelta(
        days=lookback_days
    )
    execution_mode = "INCREMENTAL"

fecha_inicio_str = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin_str = fecha_fin.strftime("%Y-%m-%d")

print(f"Modo de ejecución: {execution_mode}")
print(f"Tabla Bronze: {bronze_table}")
print(
    f"Rango solicitado a SIMEM: "
    f"{fecha_inicio_str} a {fecha_fin_str}"
)

df_generacion = ReadSIMEM(
    "055A4D",
    fecha_inicio_str,
    fecha_fin_str
).main(filter=True)

if df_generacion is None or df_generacion.empty:
    raise ValueError(
        f"SIMEM no devolvió registros entre "
        f"{fecha_inicio_str} y {fecha_fin_str}"
    )

print(f"Registros descargados: {len(df_generacion):,}")

display(df_generacion)

In [0]:
# Guardar como un solo archivo JSON usando pandas, sobrescribiendo si ya existe
# Usar método 'to_json' con compresión para acelerar escritura y reducir tamaño
df_generacion.to_json(
    "/Volumes/observatorio_dev/landing/raw_files/generacion_real.json.gz",
    orient='records',
    lines=True,
    mode='w',
    compression='gzip'
)
# Para 10,000 filas, guardar en un solo archivo JSON comprimido es eficiente y adecuado; no es necesario dividir en varios archivos.